# 02 · Pair Selection via Cointegration
### S&P 500 Pairs Trading — Research Pipeline

---

**Objective:** Reduce the full S&P 500 universe (~448 tickers) down to a set of statistically cointegrated, economically similar pairs that are viable candidates for a mean-reversion strategy.

**Screening funnel:**

```
~503 tickers
    │
    ▼  [Stock screeners]
    │  ① No missing prices
    │  ② No missing volume  
    │  ③ I(1) integration order (ADF + KPSS)
    │
    ▼  ~N surviving tickers  →  C(N,2) candidate pairs
    │
    ▼  [Similarity filter — ANY of:]
    │  ④ Same sector
    │  ⑤ Same industry
    │  ⑥ Same market-cap bucket
    │  ⑦ Same volume bucket
    │
    ▼  [Cointegration battery — ALL must pass:]
    │  ⑧ Engle-Granger (2 Step approach)
    │
    ▼  Final cointegrated pairs  →  data/output/sp500_pairs.parquet
```

## 0 · Environment Setup

In [1]:
import os
import sys
import warnings
from pathlib import Path
from itertools import combinations

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import norm
from IPython.display import display
from dateutil.relativedelta import relativedelta

from statarb.context import AppContext
from statarb.cointegration import (
    get_integration,
    pair_cointegration_tests,
    measure_mean_reversion,
)
from statarb.screener import (
    Universe,
    NoMissingPriceScreener,
    NoMissingVolumeScreener,
    IntegrationScreener,
    SameSectorScreener,
    SameIndustryScreener,
    SameMarketCapBucketScreener,
    SameVolumeBucketScreener,
    CorrelationScreener,
    DistanceScreener,
    CointegrationScreener,
    VECMRankScreener,
)

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "#F8F9FA",
    "axes.grid":         True,
    "grid.color":        "#CCCCCC",
    "grid.linewidth":    0.6,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        120,
})
PALETTE = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
pd.set_option('display.max_columns', None)

ctx = AppContext.instance(config_path=str(PROJECT_ROOT / "config.yaml"))
cfg = ctx.cfg

TRAIN_START = cfg["splits"]["train_start"]
TRAIN_END   = cfg["splits"]["train_end"]
VAL_START = cfg["splits"]["val_start"]
VAL_END   = cfg["splits"]["val_end"]
TEST_START = cfg["splits"]["test_start"]
TEST_END   = cfg["splits"]["test_end"]

## 1 · Stock Screening

Three hard filters applied at the ticker level before any pair is formed.

| Screener | Rule |
|---|---|
| `NoMissingPriceScreener` | Drops tickers with any NaN in adj-close during training |
| `NoMissingVolumeScreener` | Drops tickers with any NaN / zero volume during training |
| `IntegrationScreener` | Keeps only I(1) tickers (ADF rejects + KPSS does not reject stationarity on levels; reversed on first differences) |

In [2]:
os.makedirs(cfg["data"]["output_dir"], exist_ok=True)

df_prices_log = ctx.df_prices_log
df_tickerinfo = ctx.df_tickerinfo

# Slice to training window for all screening decisions
df_train = df_prices_log.loc[TRAIN_START:TRAIN_END]

universe = Universe(cfg, ctx)

n_before = len(universe._surviving_tickers) if hasattr(universe, "_surviving_tickers") else df_prices_log.shape[1]

universe.screen_stocks(
    NoMissingPriceScreener(cfg),
    NoMissingVolumeScreener(cfg),
    IntegrationScreener(cfg),
)
surviving = universe.surviving_tickers
print(f"Tickers before stock screening : ~{df_prices_log.shape[1]}")
print(f"Tickers after  stock screening : {len(surviving)}")

universe.create_pairs()
num_pairs = len(universe.df_pairs)
print(f"Number of pairs: {num_pairs}")

OUTPUT_DIR = Path(cfg["data"]["output_dir"])
SCREENING_FILE = OUTPUT_DIR / "sp500_stock_screening.parquet"

df_screening_results = pd.read_parquet(SCREENING_FILE)
print(f"-------------------------------- Loading screening results --------------------------------")
display(df_screening_results.head())

failed_tickers = df_screening_results[df_screening_results.all(axis=1) == False]
print(f"--------------- Example of tickers that failed at least one test ({len(failed_tickers)} total) ---------------")
display(failed_tickers.head())

Tickers before stock screening : ~503
Tickers after  stock screening : 447
Number of pairs: 99681
-------------------------------- Loading screening results --------------------------------


,ticker,no_missing_price,no_missing_volume,is_integrated
0,A,True,True,True
1,AAPL,True,True,True
2,ABBV,True,True,True
3,ABNB,False,False,False
4,ABT,True,True,True


--------------- Example of tickers that failed at least one test (56 total) ---------------


,ticker,no_missing_price,no_missing_volume,is_integrated
3,ABNB,False,False,False
25,AMCR,True,False,True
26,AMD,True,False,True
39,APP,False,False,False
63,BKR,True,False,True


In [3]:
rw_cfg = cfg["rolling_window"]
current_date = pd.to_datetime(rw_cfg["start_date"])
end_date = pd.to_datetime(rw_cfg["end_date"])
step_months = rw_cfg["trading_months"]
lookback_months = rw_cfg["formation_months"]

while current_date < end_date:
    date_str = current_date.strftime('%Y')
    combined_file = OUTPUT_DIR / f"pairs_combined_{date_str}.parquet"
    
    # 2. CHECK IF EXISTS: Skip if the combined file is already there
    if combined_file.exists():
        current_date += relativedelta(months=step_months)
        continue

    train_end = current_date - relativedelta(days=1)
    train_start = train_end - relativedelta(months=lookback_months) + relativedelta(days=1)
    
    cfg["splits"]["train_start"] = train_start.strftime("%Y-%m-%d")
    cfg["splits"]["train_end"]   = train_end.strftime("%Y-%m-%d")
    
    universe = Universe(cfg, ctx)

    # Phase 1: Stock Screening
    n_before = len(ctx.df_prices_log.columns)
    universe.screen_stocks(NoMissingPriceScreener(cfg), NoMissingVolumeScreener(cfg), IntegrationScreener(cfg))
    n_after = len(universe.surviving_tickers)
    
    # Phase 2: Candidate Generation
    universe.create_pairs()
    df_candidates = universe.df_pairs.copy()
    n_candidates = len(df_candidates)

    # --- Strategy 1: Correlation ---
    universe.df_pairs = df_candidates.copy()
    df_corr = universe.screen_pairs(CorrelationScreener(cfg))
    universe.export_to_path(OUTPUT_DIR / f"pairs_corr_{current_date.strftime('%Y')}.parquet")

    # --- Strategy 2: Distance ---
    universe.df_pairs = df_candidates.copy()
    df_dist = universe.screen_pairs(DistanceScreener(cfg))
    universe.export_to_path(OUTPUT_DIR / f"pairs_dist_{current_date.strftime('%Y')}.parquet")

    # --- Strategy 3: Cointegration ---
    universe.df_pairs = df_candidates.copy()
    df_coint = universe.screen_pairs(CointegrationScreener(cfg))
    universe.df_pairs = df_coint.copy()
    df_final_coint = universe.screen_pairs(VECMRankScreener(cfg, rank_condition=lambda x: x == 1))
    universe.export_to_path(OUTPUT_DIR / f"pairs_coint_{current_date.strftime('%Y')}.parquet")

    # --- Strategy 4: Combined ---
    universe.df_pairs = df_final_coint.copy()
    df_combined = universe.screen_pairs( 
                CorrelationScreener(cfg),
                DistanceScreener(cfg), 
                logic="all"
            )
    
    target_file = f"pairs_combined_{current_date.strftime('%Y')}.parquet"
    universe.export_to_path(OUTPUT_DIR / target_file)

    # 4. Advance
    current_date += relativedelta(months=step_months)